# Task 4.2 — Simultaneous Red, Green and Blue Detection

**Platform:** Zybo Legacy / PYNQ 3.0.1, OpenCV 4.5.4, Logitech C270 (`/dev/video0`).

**Objective:** detect red, green and blue **regions** within a single webcam frame, display their masks, mark contours and calculate centroid coordinates.

Place three red, green and blue markers in front of the camera and execute the self-contained code cell. Color regions separated by white labels may be counted individually.

In [ ]:
import cv2
import numpy as np
import time
from IPython.display import display, Image

DEVICE = "/dev/video0"
cap = cv2.VideoCapture(DEVICE, cv2.CAP_V4L2)
try:
    if not cap.isOpened():
        print("ERROR: Cannot open camera:", DEVICE)
    else:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        frame = None
        # Warm up automatic exposure before capturing the test image.
        for _ in range(30):
            ret, img = cap.read()
            if ret and img is not None:
                frame = img
            time.sleep(0.05)
        if frame is None:
            print("ERROR: No frame captured.")
        else:
            print("Frame shape:", frame.shape)
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            # HSV intervals and measured RGB dominance margins.
            COLORS = {
                "RED": {
                    "ranges": [((0, 100, 50), (10, 255, 255)),
                               ((170, 100, 50), (179, 255, 255))],
                    "margin": 35, "box": (0, 0, 255)
                },
                "GREEN": {
                    "ranges": [((35, 45, 35), (90, 255, 255))],
                    "margin": 10, "box": (0, 255, 0)
                },
                "BLUE": {
                    "ranges": [((95, 80, 50), (130, 255, 255))],
                    "margin": 35, "box": (255, 0, 0)
                }
            }
            b, g, r = [c.astype(np.int16) for c in cv2.split(frame)]
            result = frame.copy()
            masks, counts = {}, {}
            for name, config in COLORS.items():
                hsv_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
                for lower, upper in config["ranges"]:
                    hsv_mask = cv2.bitwise_or(
                        hsv_mask, cv2.inRange(hsv, lower, upper)
                    )
                margin = config["margin"]
                if name == "RED":
                    dominant = (r > g + margin) & (r > b + margin) & (r > 80)
                elif name == "GREEN":
                    dominant = (g > r + margin) & (g > b + margin) & (g > 50)
                else:
                    dominant = (b > r + margin) & (b > g + margin) & (b > 80)
                mask = cv2.bitwise_and(
                    hsv_mask, dominant.astype(np.uint8) * 255
                )
                mask = cv2.morphologyEx(
                    mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)
                )
                # Closing fills small gaps, not large white label segments.
                mask = cv2.morphologyEx(
                    mask, cv2.MORPH_CLOSE,
                    cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
                )
                masks[name] = mask
                contours, _ = cv2.findContours(
                    mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                )
                count = 0
                for contour in contours:
                    if cv2.contourArea(contour) < 150:
                        continue
                    x, y, w, h = cv2.boundingRect(contour)
                    M = cv2.moments(contour)
                    if M["m00"]:
                        cx = int(M["m10"] / M["m00"])
                        cy = int(M["m01"] / M["m00"])
                    else:
                        cx, cy = x + w // 2, y + h // 2
                    cv2.rectangle(result, (x, y), (x+w, y+h),
                                  config["box"], 2)
                    cv2.circle(result, (cx, cy), 4, config["box"], -1)
                    cv2.putText(
                        result, "{} ({},{})".format(name, cx, cy),
                        (x, max(15, y-8)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.45, config["box"], 1
                    )
                    count += 1
                counts[name] = count
            print("Detection results:")
            for name in COLORS:
                print("{}: {}".format(name, counts[name]))
            top = np.hstack([
                cv2.resize(img, (320, 240)) for img in (frame, result)
            ])
            bottom = np.hstack([
                cv2.resize(cv2.cvtColor(masks[name], cv2.COLOR_GRAY2BGR),
                           (320, 240))
                for name in ("RED", "GREEN", "BLUE")
            ])
            for heading, panel in (
                ("Original image | Detected regions", top),
                ("RED mask | GREEN mask | BLUE mask", bottom)
            ):
                print(heading)
                ok, jpg = cv2.imencode(".jpg", panel)
                if ok:
                    display(Image(data=jpg.tobytes()))
finally:
    # Release the camera on success and on errors.
    cap.release()
    print("Camera released.")

## Observed result on Zybo

With three markers, the tested configuration produced **RED: 2, GREEN: 2, BLUE: 1**. All three colors were correctly identified; red and green markers yielded two distinct colored regions because their white label areas interrupt the masks. This is acceptable for a **color-region detection** task.

## Student checks

1. Compare the three binary masks and the annotated image.
2. Record the number of detected *regions* for each color; do not equate this to the number of physical markers.
3. Change `margin` for GREEN from `10` to `35` and observe whether the green marker disappears.
4. Explain how HSV limits, RGB dominance, morphological opening/closing and minimum contour area affect the result.

Thresholds were experimentally checked with the Logitech C270 and these markers; other lighting conditions and objects can alter detection.